# Estrattore Statistiche Serie A
## Estrazione completa delle statistiche di squadre e giocatori della Serie A

Questo notebook utilizza le API di SofaScore per estrarre le statistiche complete di tutte le squadre e tutti i giocatori della Serie A italiana.

### Funzionalità:
- Estrazione dei dati di tutte le squadre della Serie A
- Raccolta delle informazioni di tutti i giocatori per ogni squadra
- Estrazione delle statistiche dettagliate per ogni giocatore
- Salvataggio dei dati in formato DataFrame per analisi successive

### Fonte dati:
- **API SofaScore**: https://www.sofascore.com/api/
- **Campionato**: Serie A (Tournament ID: 23)
- **Stagione**: 2024/25 (Season ID: 76457)

In [ ]:
# Import delle librerie necessarie
import requests
import pandas as pd
import json
import time
from datetime import datetime
import numpy as np
from typing import Dict, List, Optional
import warnings
warnings.filterwarnings('ignore')

print("Librerie importate con successo!")
print(f"Timestamp di avvio: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

In [ ]:
# Configurazione API SofaScore
class SofaScoreAPI:
    def __init__(self):
        self.base_url = "https://www.sofascore.com/api/v1"
        self.headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36',
            'Accept': 'application/json',
            'Accept-Language': 'it-IT,it;q=0.9,en;q=0.8'
        }
        self.serie_a_tournament_id = 23
        self.current_season_id = 76457  # Serie A 2024/25
        self.request_delay = 0.5  # Delay tra le richieste per evitare rate limiting
        
    def make_request(self, endpoint: str, retries: int = 3) -> Optional[Dict]:
        """Effettua una richiesta HTTP con retry automatico"""
        for attempt in range(retries):
            try:
                url = f"{self.base_url}{endpoint}"
                response = requests.get(url, headers=self.headers, timeout=10)
                
                if response.status_code == 200:
                    time.sleep(self.request_delay)  # Rate limiting
                    return response.json()
                elif response.status_code == 429:  # Too Many Requests
                    wait_time = 2 ** attempt  # Exponential backoff
                    print(f"Rate limit raggiunto. Attesa {wait_time} secondi...")
                    time.sleep(wait_time)
                else:
                    print(f"Errore HTTP {response.status_code} per {endpoint}")
                    
            except requests.exceptions.RequestException as e:
                print(f"Errore di rete per {endpoint}: {e}")
                if attempt < retries - 1:
                    time.sleep(2 ** attempt)
                    
        return None

# Inizializzazione dell'API client
api = SofaScoreAPI()
print("Configurazione API completata!")
print(f"URL base: {api.base_url}")
print(f"Tournament ID Serie A: {api.serie_a_tournament_id}")
print(f"Season ID: {api.current_season_id}")

In [ ]:
# Estrazione dei dati delle squadre della Serie A
def get_serie_a_teams() -> List[Dict]:
    """Estrae tutte le squadre della Serie A dalla classifica"""
    print("Estrazione delle squadre della Serie A...")
    
    endpoint = f"/unique-tournament/{api.serie_a_tournament_id}/season/{api.current_season_id}/standings/total"
    data = api.make_request(endpoint)
    
    teams = []
    if data and 'standings' in data:
        for standing in data['standings']:
            if standing.get('type') == 'total' and 'rows' in standing:
                for row in standing['rows']:
                    team_info = row.get('team', {})
                    team_data = {
                        'team_id': team_info.get('id'),
                        'name': team_info.get('name'),
                        'short_name': team_info.get('shortName'),
                        'name_code': team_info.get('nameCode'),
                        'slug': team_info.get('slug'),
                        'position': row.get('position'),
                        'points': row.get('points'),
                        'matches': row.get('matches'),
                        'wins': row.get('wins'),
                        'draws': row.get('draws'),
                        'losses': row.get('losses'),
                        'goals_for': row.get('scoresFor'),
                        'goals_against': row.get('scoresAgainst'),
                        'goal_difference': row.get('goalDifference')
                    }
                    teams.append(team_data)
    
    print(f"Trovate {len(teams)} squadre della Serie A")
    return teams

# Estrazione delle squadre
serie_a_teams = get_serie_a_teams()

# Creazione del DataFrame delle squadre
teams_df = pd.DataFrame(serie_a_teams)
print("\nDataFrame delle squadre creato:")
print(teams_df[['name', 'position', 'points', 'matches']].head(10))

In [ ]:
# Estrazione degli ID dei giocatori per ogni squadra
def get_team_players(team_id: int, team_name: str) -> List[Dict]:
    """Estrae tutti i giocatori di una squadra"""
    print(f"Estrazione giocatori per {team_name}...")
    
    endpoint = f"/team/{team_id}/players"
    data = api.make_request(endpoint)
    
    players = []
    if data and 'players' in data:
        for player_data in data['players']:
            player_info = player_data.get('player', {})
            player = {
                'player_id': player_info.get('id'),
                'team_id': team_id,
                'team_name': team_name,
                'name': player_info.get('name'),
                'short_name': player_info.get('shortName'),
                'slug': player_info.get('slug'),
                'position': player_info.get('position'),
                'jersey_number': player_info.get('jerseyNumber'),
                'height': player_info.get('height'),
                'date_of_birth': player_info.get('dateOfBirth'),
                'preferred_foot': player_info.get('preferredFoot'),
                'market_value': player_info.get('proposedMarketValue'),
                'nationality': player_info.get('country', {}).get('name') if player_info.get('country') else None
            }
            players.append(player)
    
    print(f"  Trovati {len(players)} giocatori per {team_name}")
    return players

# Estrazione di tutti i giocatori della Serie A
all_players = []
total_teams = len(serie_a_teams)

print("Inizio estrazione giocatori per tutte le squadre...")
print(f"Squadre da processare: {total_teams}")
print("-" * 50)

for i, team in enumerate(serie_a_teams, 1):
    team_id = team['team_id']
    team_name = team['name']
    
    print(f"[{i}/{total_teams}] Processando {team_name}...")
    
    if team_id:
        team_players = get_team_players(team_id, team_name)
        all_players.extend(team_players)
    
    # Progresso ogni 5 squadre
    if i % 5 == 0:
        print(f"Progresso: {i}/{total_teams} squadre completate")
        print(f"Giocatori raccolti finora: {len(all_players)}")
        print("-" * 30)

print(f"\nEstrazione completata!")
print(f"Totale giocatori raccolti: {len(all_players)}")

# Creazione DataFrame dei giocatori
players_df = pd.DataFrame(all_players)
print(f"\nDataFrame giocatori creato con {len(players_df)} righe")
print("\nPrime 10 righe:")
print(players_df[['name', 'team_name', 'position', 'jersey_number']].head(10))

In [ ]:
# Estrazione delle statistiche dettagliate per ogni giocatore
def get_player_statistics(player_id: int, player_name: str) -> List[Dict]:
    """Estrae le statistiche complete di un giocatore"""
    endpoint = f"/player/{player_id}/statistics"
    data = api.make_request(endpoint)
    
    statistics = []
    if data and 'seasons' in data:
        for season in data['seasons']:
            stats = season.get('statistics', {})
            tournament = season.get('uniqueTournament', {})
            team = season.get('team', {})
            season_info = season.get('season', {})
            
            # Filtra per mantenere solo le statistiche della Serie A
            if tournament.get('id') == api.serie_a_tournament_id:
                stat_record = {
                    'player_id': player_id,
                    'player_name': player_name,
                    'season': season.get('year'),
                    'tournament': tournament.get('name'),
                    'team_name': team.get('name'),
                    
                    # Statistiche offensive
                    'goals': stats.get('goals', 0),
                    'assists': stats.get('assists', 0),
                    'goals_assists_sum': stats.get('goalsAssistsSum', 0),
                    'expected_goals': stats.get('expectedGoals', 0),
                    'expected_assists': stats.get('expectedAssists', 0),
                    'big_chances_created': stats.get('bigChancesCreated', 0),
                    'big_chances_missed': stats.get('bigChancesMissed', 0),
                    'shots_on_target': stats.get('shotsOnTarget', 0),
                    'total_shots': stats.get('totalShots', 0),
                    'shots_from_inside_box': stats.get('shotsFromInsideTheBox', 0),
                    'key_passes': stats.get('keyPasses', 0),
                    'pass_to_assist': stats.get('passToAssist', 0),
                    
                    # Statistiche di gioco
                    'appearances': stats.get('appearances', 0),
                    'minutes_played': stats.get('minutesPlayed', 0),
                    'rating': stats.get('rating', 0),
                    'total_rating': stats.get('totalRating', 0),
                    'count_rating': stats.get('countRating', 0),
                    
                    # Statistiche di passaggio
                    'accurate_passes': stats.get('accuratePasses', 0),
                    'total_passes': stats.get('totalPasses', 0),
                    'accurate_passes_percentage': stats.get('accuratePassesPercentage', 0),
                    'accurate_long_balls': stats.get('accurateLongBalls', 0),
                    'total_long_balls': stats.get('totalLongBalls', 0),
                    'accurate_crosses': stats.get('accurateCrosses', 0),
                    'total_cross': stats.get('totalCross', 0),
                    
                    # Statistiche difensive
                    'tackles': stats.get('tackles', 0),
                    'interceptions': stats.get('interceptions', 0),
                    'blocked_shots': stats.get('blockedShots', 0),
                    'outfielder_blocks': stats.get('outfielderBlocks', 0),
                    'clean_sheet': stats.get('cleanSheet', 0),
                    'goals_conceded': stats.get('goalsConceded', 0),
                    'saves': stats.get('saves', 0),
                    'successful_dribbles': stats.get('successfulDribbles', 0),
                    'dribbled_past': stats.get('dribbledPast', 0),
                    'aerial_duels_won': stats.get('aerialDuelsWon', 0),
                    
                    # Cartellini
                    'yellow_cards': stats.get('yellowCards', 0),
                    'red_cards': stats.get('redCards', 0),
                    'error_lead_to_goal': stats.get('errorLeadToGoal', 0)
                }
                statistics.append(stat_record)
    
    return statistics

# Estrazione delle statistiche per tutti i giocatori
print("Inizio estrazione statistiche per tutti i giocatori...")
print(f"Giocatori da processare: {len(all_players)}")
print("ATTENZIONE: Questo processo può richiedere molto tempo!")
print("-" * 50)

all_statistics = []
failed_players = []
total_players = len(all_players)

# Processa i giocatori in batch per monitorare il progresso
for i, player in enumerate(all_players, 1):
    player_id = player['player_id']
    player_name = player['name']
    
    if player_id:
        try:
            player_stats = get_player_statistics(player_id, player_name)
            all_statistics.extend(player_stats)
            
            # Progresso ogni 50 giocatori
            if i % 50 == 0:
                print(f"Progresso: {i}/{total_players} giocatori processati")
                print(f"Statistiche raccolte: {len(all_statistics)}")
                print(f"Giocatori falliti: {len(failed_players)}")
                print("-" * 30)
                
        except Exception as e:
            print(f"Errore per {player_name}: {e}")
            failed_players.append(player_name)
    else:
        failed_players.append(player_name)

print(f"\nEstrazione statistiche completata!")
print(f"Statistiche raccolte: {len(all_statistics)}")
print(f"Giocatori con errori: {len(failed_players)}")

if failed_players:
    print(f"\nGiocatori falliti: {failed_players[:10]}...")  # Mostra solo i primi 10

In [ ]:
# Elaborazione e pulizia dei dati statistici
def clean_and_process_data():
    """Pulisce e elabora i dati raccolti"""
    print("Elaborazione e pulizia dei dati...")
    
    # Creazione DataFrame delle statistiche
    statistics_df = pd.DataFrame(all_statistics)
    
    if len(statistics_df) > 0:
        # Conversione dei tipi di dati
        numeric_columns = [
            'goals', 'assists', 'goals_assists_sum', 'expected_goals', 'expected_assists',
            'big_chances_created', 'big_chances_missed', 'shots_on_target', 'total_shots',
            'shots_from_inside_box', 'key_passes', 'appearances', 'minutes_played',
            'rating', 'accurate_passes', 'total_passes', 'accurate_passes_percentage',
            'tackles', 'interceptions', 'blocked_shots', 'clean_sheet', 'goals_conceded',
            'saves', 'successful_dribbles', 'dribbled_past', 'aerial_duels_won',
            'yellow_cards', 'red_cards'
        ]
        
        for col in numeric_columns:
            if col in statistics_df.columns:
                statistics_df[col] = pd.to_numeric(statistics_df[col], errors='coerce').fillna(0)
        
        # Calcolo di statistiche derivate
        statistics_df['goals_per_match'] = np.where(
            statistics_df['appearances'] > 0,
            statistics_df['goals'] / statistics_df['appearances'],
            0
        )
        
        statistics_df['assists_per_match'] = np.where(
            statistics_df['appearances'] > 0,
            statistics_df['assists'] / statistics_df['appearances'],
            0
        )
        
        statistics_df['minutes_per_match'] = np.where(
            statistics_df['appearances'] > 0,
            statistics_df['minutes_played'] / statistics_df['appearances'],
            0
        )
        
        statistics_df['pass_accuracy'] = np.where(
            statistics_df['total_passes'] > 0,
            statistics_df['accurate_passes'] / statistics_df['total_passes'] * 100,
            0
        )
        
        statistics_df['shot_accuracy'] = np.where(
            statistics_df['total_shots'] > 0,
            statistics_df['shots_on_target'] / statistics_df['total_shots'] * 100,
            0
        )
        
        print(f"DataFrame statistiche creato con {len(statistics_df)} righe")
        return statistics_df
    else:
        print("Nessuna statistica disponibile!")
        return pd.DataFrame()

# Elaborazione dei dati
statistics_df = clean_and_process_data()

# Pulizia DataFrame giocatori
if len(players_df) > 0:
    # Rimozione duplicati basati su player_id
    players_df = players_df.drop_duplicates(subset=['player_id'])
    
    # Pulizia dei valori nulli
    players_df['jersey_number'] = pd.to_numeric(players_df['jersey_number'], errors='coerce')
    players_df['height'] = pd.to_numeric(players_df['height'], errors='coerce')
    players_df['market_value'] = pd.to_numeric(players_df['market_value'], errors='coerce')
    
    print(f"DataFrame giocatori pulito: {len(players_df)} giocatori unici")

# Pulizia DataFrame squadre
if len(teams_df) > 0:
    # Ordinamento per posizione in classifica
    teams_df = teams_df.sort_values('position').reset_index(drop=True)
    print(f"DataFrame squadre ordinato: {len(teams_df)} squadre")

print("\nPulizia dati completata!")
print(f"- Squadre: {len(teams_df) if 'teams_df' in locals() else 0}")
print(f"- Giocatori: {len(players_df) if 'players_df' in locals() else 0}")
print(f"- Statistiche: {len(statistics_df) if 'statistics_df' in locals() else 0}")

In [ ]:
# Creazione di DataFrame comprensivi e analisi
def create_comprehensive_dataframes():
    """Crea DataFrame completi combinando tutti i dati"""
    print("Creazione DataFrame comprensivi...")
    
    # DataFrame completo giocatori con statistiche
    if len(statistics_df) > 0 and len(players_df) > 0:
        # Merge delle statistiche con i dati dei giocatori
        complete_players_df = players_df.merge(
            statistics_df,
            on='player_id',
            how='left'
        )
        
        print(f"DataFrame completo giocatori: {len(complete_players_df)} righe")
    else:
        complete_players_df = players_df.copy()
        print("Merge non possibile - usando solo dati giocatori")
    
    # Statistiche aggregate per squadra
    if len(statistics_df) > 0:
        team_stats = statistics_df.groupby('team_name').agg({
            'goals': 'sum',
            'assists': 'sum',
            'appearances': 'sum',
            'minutes_played': 'sum',
            'yellow_cards': 'sum',
            'red_cards': 'sum',
            'rating': 'mean',
            'goals_per_match': 'mean',
            'assists_per_match': 'mean',
            'pass_accuracy': 'mean',
            'shot_accuracy': 'mean'
        }).round(2)
        
        # Merge con dati squadre
        complete_teams_df = teams_df.merge(
            team_stats.reset_index(),
            left_on='name',
            right_on='team_name',
            how='left'
        )
        
        print(f"DataFrame completo squadre: {len(complete_teams_df)} righe")
    else:
        complete_teams_df = teams_df.copy()
        print("Statistiche squadre non disponibili")
    
    return complete_players_df, complete_teams_df

# Creazione DataFrame finali
complete_players_df, complete_teams_df = create_comprehensive_dataframes()

# Analisi esplorativa dei dati
print("\n" + "="*50)
print("ANALISI ESPLORATIVA DEI DATI")
print("="*50)

# Analisi squadre
if len(complete_teams_df) > 0:
    print("\n🏆 CLASSIFICA SERIE A:")
    print(complete_teams_df[['position', 'name', 'points', 'matches', 'goal_difference']].head(10))
    
    print(f"\n📊 STATISTICHE SQUADRE:")
    if 'goals' in complete_teams_df.columns:
        print(f"- Squadra con più gol: {complete_teams_df.loc[complete_teams_df['goals'].idxmax(), 'name']}")
        print(f"- Squadra con più assist: {complete_teams_df.loc[complete_teams_df['assists'].idxmax(), 'name']}")

# Analisi giocatori
if len(complete_players_df) > 0:
    print(f"\n👥 GIOCATORI:")
    print(f"- Totale giocatori: {len(complete_players_df)}")
    
    # Distribuzione per ruolo
    if 'position' in complete_players_df.columns:
        position_counts = complete_players_df['position'].value_counts()
        print(f"- Distribuzione per ruolo:")
        for pos, count in position_counts.head().items():
            print(f"  {pos}: {count}")
    
    # Top scorer se disponibili le statistiche
    if 'goals' in complete_players_df.columns:
        top_scorers = complete_players_df.nlargest(10, 'goals')[['name', 'team_name', 'goals', 'assists']]
        print(f"\n⚽ TOP 10 MARCATORI:")
        print(top_scorers)

# Statistiche dataset
print(f"\n📈 STATISTICHE DATASET:")
print(f"- Squadre: {len(complete_teams_df)}")
print(f"- Giocatori: {len(complete_players_df)}")
if len(statistics_df) > 0:
    print(f"- Record statistiche: {len(statistics_df)}")
    print(f"- Stagioni coperte: {sorted(statistics_df['season'].unique()) if 'season' in statistics_df.columns else 'N/A'}")

print("\nDataFrame comprensivi creati con successo!")

In [ ]:
# Salvataggio dei dati in file
def save_dataframes_to_files():
    """Salva tutti i DataFrame in diversi formati"""
    print("Salvataggio dei dati in file...")
    
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    base_filename = f"serie_a_data_{timestamp}"
    
    saved_files = []
    
    try:
        # Salvataggio DataFrame squadre
        if len(complete_teams_df) > 0:
            # CSV
            teams_csv = f"{base_filename}_squadre.csv"
            complete_teams_df.to_csv(teams_csv, index=False, encoding='utf-8')
            saved_files.append(teams_csv)
            
            # JSON
            teams_json = f"{base_filename}_squadre.json"
            complete_teams_df.to_json(teams_json, orient='records', indent=2, force_ascii=False)
            saved_files.append(teams_json)
            
            print(f"✅ Squadre salvate: {len(complete_teams_df)} righe")
        
        # Salvataggio DataFrame giocatori
        if len(complete_players_df) > 0:
            # CSV
            players_csv = f"{base_filename}_giocatori.csv"
            complete_players_df.to_csv(players_csv, index=False, encoding='utf-8')
            saved_files.append(players_csv)
            
            # JSON (limitato ai primi 1000 per dimensioni)
            players_json = f"{base_filename}_giocatori_sample.json"
            complete_players_df.head(1000).to_json(players_json, orient='records', indent=2, force_ascii=False)
            saved_files.append(players_json)
            
            print(f"✅ Giocatori salvati: {len(complete_players_df)} righe")
        
        # Salvataggio DataFrame statistiche dettagliate
        if len(statistics_df) > 0:
            # CSV
            stats_csv = f"{base_filename}_statistiche.csv"
            statistics_df.to_csv(stats_csv, index=False, encoding='utf-8')
            saved_files.append(stats_csv)
            
            print(f"✅ Statistiche salvate: {len(statistics_df)} righe")
        
        # Salvataggio DataFrame combinato (pickle per preservare tutti i tipi)
        if len(complete_players_df) > 0:
            combined_pickle = f"{base_filename}_completo.pkl"
            complete_players_df.to_pickle(combined_pickle)
            saved_files.append(combined_pickle)
            
            print(f"✅ Dataset completo salvato (pickle): {len(complete_players_df)} righe")
        
        # Creazione file di metadati
        metadata = {
            'extraction_timestamp': datetime.now().isoformat(),
            'serie_a_season': '2024/25',
            'tournament_id': api.serie_a_tournament_id,
            'season_id': api.current_season_id,
            'total_teams': len(complete_teams_df) if len(complete_teams_df) > 0 else 0,
            'total_players': len(complete_players_df) if len(complete_players_df) > 0 else 0,
            'total_statistics_records': len(statistics_df) if len(statistics_df) > 0 else 0,
            'failed_players': len(failed_players) if 'failed_players' in locals() else 0,
            'files_created': saved_files
        }
        
        metadata_file = f"{base_filename}_metadata.json"
        with open(metadata_file, 'w', encoding='utf-8') as f:
            json.dump(metadata, f, indent=2, ensure_ascii=False)
        saved_files.append(metadata_file)
        
        print(f"✅ Metadati salvati")
        
        print(f"\n🎉 SALVATAGGIO COMPLETATO!")
        print(f"File creati: {len(saved_files)}")
        for file in saved_files:
            print(f"  - {file}")
            
        return saved_files
        
    except Exception as e:
        print(f"❌ Errore durante il salvataggio: {e}")
        return []

# Esecuzione del salvataggio
saved_files = save_dataframes_to_files()

# Riepilogo finale
print("\n" + "="*60)
print("🏁 ESTRAZIONE SERIE A COMPLETATA!")
print("="*60)
print(f"📅 Data e ora: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"🏆 Campionato: Serie A 2024/25")
print(f"📊 Dati estratti:")
print(f"   • Squadre: {len(complete_teams_df) if 'complete_teams_df' in locals() else 0}")
print(f"   • Giocatori: {len(complete_players_df) if 'complete_players_df' in locals() else 0}")
print(f"   • Statistiche: {len(statistics_df) if 'statistics_df' in locals() else 0}")
print(f"💾 File salvati: {len(saved_files)}")
print("="*60)

# Istruzioni per l'uso dei dati
print("\n📖 COME UTILIZZARE I DATI:")
print("1. CSV files: Apribili con Excel, Google Sheets, o pandas")
print("2. JSON files: Utilizzabili in applicazioni web o per API")
print("3. Pickle files: Per uso avanzato con pandas (preserva tutti i tipi di dati)")
print("\nEsempio di caricamento:")
print("  import pandas as pd")
print(f"  df = pd.read_csv('{saved_files[0] if saved_files else 'filename.csv'}')")
print("\nBuona analisi! ⚽📊")

## 📋 Riepilogo e Istruzioni d'Uso

### Dati Estratti
Questo notebook ha estratto i seguenti dati dalla Serie A 2024/25:

1. **Squadre**: Informazioni complete di tutte le 20 squadre della Serie A
   - Dati di classifica (posizione, punti, partite)
   - Statistiche aggregate dei giocatori

2. **Giocatori**: Roster completo di tutti i giocatori
   - Informazioni anagrafiche e contrattuali
   - Ruolo, numero di maglia, nazionalità

3. **Statistiche**: Dati dettagliati delle prestazioni
   - Gol, assist, presenze, minuti giocati
   - Statistiche di passaggio, tiri, difesa
   - Cartellini e altre metriche

### File Generati
- **CSV**: Per analisi in Excel o altri strumenti
- **JSON**: Per applicazioni web e API
- **Pickle**: Per uso avanzato con pandas

### Possibili Utilizzi
- 📊 **Analisi prestazioni**: Confronto tra giocatori e squadre
- 🎯 **Fantacalcio**: Valutazione giocatori per aste
- 📈 **Data Science**: Modelli predittivi e machine learning
- 🖥️ **Applicazioni Web**: Dashboard e visualizzazioni
- 📰 **Giornalismo Sportivo**: Statistiche per articoli

### Note Tecniche
- I dati provengono dall'API SofaScore
- Include rate limiting per rispettare i limiti del servizio
- Gestione errori automatica con retry
- Pulizia e validazione automatica dei dati

### Aggiornamenti
Per aggiornare i dati, esegui nuovamente tutte le celle del notebook. Si consiglia di farlo settimanalmente durante la stagione per avere statistiche aggiornate.